In [10]:
import fastf1
import pandas as pd
import numpy as np
from pprint import pprint

# 禁用响应警告
import warnings
warnings.filterwarnings('ignore')

print("fastf1 版本:", fastf1.__version__)

fastf1 版本: 3.8.1


In [11]:

# ============================================================
# Cell A — 配置参数 & 加载会话基础数据
# ============================================================
import os

# ── 全局配置 ─────────────────────────────────────────────────
YEAR         = 2024
ROUND        = 1          # 整数站次或字符串名称
SESSION_TYPE = 'R'        # 正赛

# 判定阈值（可调节）
TEAM_ORDER_THROTTLE_THRESHOLD   = 10.0   # Throttle_Drop_Index > 该值则触发
TEAM_ORDER_BRAKE_THRESHOLD      = 150.0  # Brake_Pressure_Delta(ms) > 该值则触发
TEAM_ORDER_CONFIDENCE_THRESHOLD = 0.5

OFF_TRACK_VMIN_THRESHOLD        = 5.0    # Vmin_Deviation(%) > 该值则触发
OFF_TRACK_CONFIDENCE_THRESHOLD  = 0.5

TELEMETRY_HZ = 10          # 按需 telemetry 重采样频率

# ── 缓存初始化 ────────────────────────────────────────────────
cache_dir = os.path.join(os.getcwd(), "cache")
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

# ── 加载 Session（基础层：不含 telemetry）────────────────────
session = fastf1.get_session(YEAR, ROUND, SESSION_TYPE)
session.load(laps=True, telemetry=False, weather=False, messages=True)

print(f"赛事: {session.event['EventName']} {YEAR}")
print(f"赛道: {session.event['Location']}")
print(f"计划总圈数: {session.total_laps}")
print(f"参赛车手: {len(session.drivers)} 名")


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


赛事: Bahrain Grand Prix 2024
赛道: Sakhir
计划总圈数: 57
参赛车手: 20 名


In [12]:

# ============================================================
# Cell B — Phase B: 同圈超车候选检测 + 排除规则
#          → 产出 full_overtakes 基础版（无 telemetry 特征）
# ============================================================

# ── B0: 构建基础 laps 表 ──────────────────────────────────────
laps_base = session.laps[[
    'LapNumber', 'Driver', 'DriverNumber', 'Team',
    'Position', 'PitInTime', 'PitOutTime', 'TrackStatus',
    'LapStartTime', 'Time',
    'FastF1Generated', 'IsAccurate',
]].copy()
laps_base['LapNumber'] = laps_base['LapNumber'].astype(int)

# ── B1: 构建圈末位次矩阵 & 圈初位次矩阵 ──────────────────────
# 圈末位次：每圈 Time 锚点的 Position
lap_end_pos = (
    laps_base.dropna(subset=['Position'])
    .assign(Position=lambda d: d['Position'].astype(int))
    .pivot_table(index='LapNumber', columns='Driver', values='Position', aggfunc='first')
    .sort_index()
)
# 圈初位次 = 上一圈圈末位次（shift 向下一行）
lap_start_pos = lap_end_pos.shift(1)

# ── B2: DNF 车手识别（保守口径：results ∩ laps 双重证据）──────
results_df = session.results[['DriverNumber', 'Status', 'Abbreviation']].copy()
dnf_by_results = set(
    results_df.loc[
        ~results_df['Status'].str.contains(r'(?i)finished|laps?$', regex=True, na=False),
        'Abbreviation'
    ]
)
last_pos = laps_base.sort_values('LapNumber').groupby('Driver').last()['Position']
dnf_by_laps = set(last_pos[last_pos.isna()].index)
dnf_drivers = dnf_by_results & dnf_by_laps

# ── B3: 进站信息快查函数 ──────────────────────────────────────
def _pit_sets(lap_num: int):
    rows = laps_base[laps_base['LapNumber'] == lap_num]
    return (
        set(rows.loc[rows['PitInTime'].notna(),  'Driver']),
        set(rows.loc[rows['PitOutTime'].notna(), 'Driver']),
    )

# ── B4: 车手元信息 ────────────────────────────────────────────
driver_meta = (
    laps_base[['Driver', 'DriverNumber', 'Team']]
    .drop_duplicates('Driver')
    .set_index('Driver')
)

# ── B5: 同圈候选遍历 + 排除 ──────────────────────────────────
records = []
sorted_laps = sorted(lap_end_pos.index)

_EMPTY_META = pd.Series({'DriverNumber': '', 'Team': ''})

for lap_n in sorted_laps:
    if lap_n == 1:                             # Lap 1 硬排除
        continue
    if lap_n not in lap_start_pos.index:       # 无前圈数据
        continue

    pos_start = lap_start_pos.loc[lap_n].dropna()
    pos_end   = lap_end_pos.loc[lap_n].dropna()
    common    = pos_start.index.intersection(pos_end.index)
    if len(common) < 2:
        continue

    lap_rows        = laps_base[laps_base['LapNumber'] == lap_n]
    t_window_start  = lap_rows['LapStartTime'].min()
    t_window_end    = lap_rows['Time'].min()

    pit_in,      pit_out      = _pit_sets(lap_n)
    pit_in_prev, pit_out_prev = _pit_sets(lap_n - 1)

    for driver_a in common:
        p_start_a = pos_start[driver_a]
        p_end_a   = pos_end[driver_a]
        if p_end_a >= p_start_a:               # 名次未改善
            continue
        if driver_a in pit_out or driver_a in pit_out_prev:   # A 出站
            continue

        meta_a = driver_meta.loc[driver_a] if driver_a in driver_meta.index else _EMPTY_META

        for driver_b in common:
            if driver_b == driver_a:
                continue
            p_start_b = pos_start[driver_b]
            p_end_b   = pos_end[driver_b]

            # 同圈位次互换：A 圈初落后 B，圈末领先 B
            if not (p_start_a > p_start_b and p_end_a < p_end_b):
                continue

            pit_inf = driver_b in pit_in or driver_b in pit_in_prev
            dnf_inf = driver_b in dnf_drivers
            if pit_inf or dnf_inf:             # 排除被动提升
                continue

            meta_b = driver_meta.loc[driver_b] if driver_b in driver_meta.index else _EMPTY_META

            records.append({
                # 统一基础字段
                'LapNumber':           lap_n,
                'Overtaker':           driver_a,
                'OvertakerNumber':     meta_a['DriverNumber'],
                'OvertakerTeam':       meta_a['Team'],
                'PositionBefore':      int(p_start_a),
                'PositionAfter':       int(p_end_a),
                'Overtaken':           driver_b,
                'OvertakenNumber':     meta_b['DriverNumber'],
                'OvertakenTeam':       meta_b['Team'],
                'SessionTime':         t_window_end,
                # 同圈窗口边界（审计）
                'SameLapWindowStart':  t_window_start,
                'SameLapWindowEnd':    t_window_end,
                # 排除审计标记（已排除，留存方便 debug）
                'PitInfluenceFlag':    False,
                'DNFInfluenceFlag':    False,
                # 基础判定字段
                'IsTeammate':       meta_a['Team'] == meta_b['Team'],
                # telemetry 特征占位 —— Phase C/D 填充
                'Is_DRS_Active':       np.nan,
                'RelativeDistance':    np.nan,
                'Throttle_Drop_Index': np.nan,
                'Brake_Pressure_Delta':np.nan,
                'TeamOrderConfidence': np.nan,
                'Vmin_Deviation':      np.nan,
                'OffTrack_Flag':       False,
                'Lockup_Flag':         False,
                'LocalYellowFlag':     False,
                'OffTrackConfidence':  np.nan,
            })

full_overtakes = pd.DataFrame(records)
print(f"Phase B 完成: {len(full_overtakes)} 条候选同圈超车（人对级别）")
print(f"  同队候选: {full_overtakes['IsTeammate'].sum()} 条")
if not full_overtakes.empty:
    print(f"  DNF 车手: {dnf_drivers or '无'}")
    print(f"  涉及圈数: {sorted(full_overtakes['LapNumber'].unique())}")


Phase B 完成: 24 条候选同圈超车（人对级别）
  同队候选: 4 条
  DNF 车手: 无
  涉及圈数: [np.int64(2), np.int64(3), np.int64(5), np.int64(7), np.int64(10), np.int64(17), np.int64(18), np.int64(21), np.int64(33), np.int64(34), np.int64(36), np.int64(38), np.int64(39), np.int64(40), np.int64(44), np.int64(46), np.int64(48), np.int64(50), np.int64(52)]


In [13]:

# ============================================================
# Cell C — Phase C: 按需 telemetry 加载与 10Hz 重采样
#          只对 full_overtakes 候选圈的两侧车手加载数据
# ============================================================

# 1. 确定候选范围
candidate_laps    = set(full_overtakes['LapNumber'].unique()) if not full_overtakes.empty else set()
candidate_drivers_a = set(full_overtakes['Overtaker'].unique()) if not full_overtakes.empty else set()
candidate_drivers_b = set(full_overtakes['Overtaken'].unique())  if not full_overtakes.empty else set()
candidate_drivers   = candidate_drivers_a | candidate_drivers_b

print(f"候选圈: {sorted(candidate_laps)}")
print(f"候选车手: {len(candidate_drivers)} 名")

if candidate_laps:
    # 2. 补加载 telemetry（仅此次调用，启用缓存后很快）
    session.load(laps=False, telemetry=True, weather=False, messages=False)

    # 3. 构建 tel_store: key=(DriverCode, LapNumber) -> Telemetry DataFrame
    tel_store: dict = {}

    for drv in candidate_drivers:
        # 根据 driver 3-letter code 查找圈数据
        drv_laps = session.laps.pick_drivers(drv)
        for lap_n in candidate_laps:
            lap_rows = drv_laps[drv_laps['LapNumber'] == lap_n]
            if lap_rows.empty:
                continue
            lap_obj = lap_rows.iloc[0]
            try:
                # 单圈 car_data + pos_data 合并
                car = lap_obj.get_car_data().add_distance().add_relative_distance()
                pos = lap_obj.get_pos_data()
                tel = car.merge_channels(pos, frequency=TELEMETRY_HZ)
                tel_store[(drv, lap_n)] = tel
            except Exception as e:
                tel_store[(drv, lap_n)] = None   # 记录缺失，后续用 NaN 填充

    loaded = sum(1 for v in tel_store.values() if v is not None)
    print(f"telemetry 加载完成: {loaded}/{len(tel_store)} 条（含失败记录）")
else:
    tel_store = {}
    print("无候选超车，跳过 telemetry 加载。")


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for car_data


候选圈: [np.int64(2), np.int64(3), np.int64(5), np.int64(7), np.int64(10), np.int64(17), np.int64(18), np.int64(21), np.int64(33), np.int64(34), np.int64(36), np.int64(38), np.int64(39), np.int64(40), np.int64(44), np.int64(46), np.int64(48), np.int64(50), np.int64(52)]
候选车手: 17 名


req            INFO 	Using cached data for position_data
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


telemetry 加载完成: 323/323 条（含失败记录）


In [14]:

# ============================================================
# Cell D — Phase D: 全量特征计算 → 填充 full_overtakes 超集 schema
# ============================================================

# ── 辅助：从 tel_store 取单圈 telemetry ──────────────────────
def _get_tel(driver: str, lap_n: int):
    t = tel_store.get((driver, lap_n))
    return t if (t is not None and len(t) > 0) else None


# ── D1: Is_DRS_Active ─────────────────────────────────────────
# DRS 值含义：0=不可用,8=可用(DRS关),10=DRS开启
# 取超车发起者（A）在该圈任意时刻 DRS=10 则视为激活
def _calc_drs(driver_a: str, lap_n: int) -> bool:
    t = _get_tel(driver_a, lap_n)
    if t is None or 'DRS' not in t.columns:
        return np.nan
    return bool((t['DRS'] == 10).any())


# ── D2: RelativeDistance ──────────────────────────────────────
# 取超车发起者（A）该圈 RelativeDistance 的中位数作为代表值
def _calc_rel_dist(driver_a: str, lap_n: int) -> float:
    t = _get_tel(driver_a, lap_n)
    if t is None or 'RelativeDistance' not in t.columns:
        return np.nan
    return float(t['RelativeDistance'].median())


# ── D3: Throttle_Drop_Index ──────────────────────────────────
# 被超车手(B)在关键加速段(40-80% RelativeDistance)的收油幅度
# = median(A.Throttle - B.Throttle) 在对齐时间窗内
def _calc_throttle_drop(driver_a: str, driver_b: str, lap_n: int) -> float:
    ta = _get_tel(driver_a, lap_n)
    tb = _get_tel(driver_b, lap_n)
    if ta is None or tb is None:
        return np.nan
    if 'Throttle' not in ta.columns or 'Throttle' not in tb.columns:
        return np.nan
    try:
        # 以 SessionTime 对齐
        ma = ta[['SessionTime', 'Throttle', 'RelativeDistance']].dropna()
        mb = tb[['SessionTime', 'Throttle']].dropna()
        merged = pd.merge_asof(
            ma.sort_values('SessionTime'),
            mb.sort_values('SessionTime'),
            on='SessionTime', direction='nearest',
            suffixes=('_a', '_b')
        )
        seg = merged.query('0.4 <= RelativeDistance <= 0.8') if 'RelativeDistance' in merged.columns else merged
        if len(seg) < 5:
            return np.nan
        return float(np.abs(np.median(seg['Throttle_a'] - seg['Throttle_b'])))
    except Exception:
        return np.nan


# ── D4: Brake_Pressure_Delta ─────────────────────────────────
# 刹车时序差异：检测 Brake=0→1 的上升沿事件，取两车对应事件时间差中位数(ms)
def _calc_brake_delta(driver_a: str, driver_b: str, lap_n: int) -> float:
    ta = _get_tel(driver_a, lap_n)
    tb = _get_tel(driver_b, lap_n)
    if ta is None or tb is None:
        return np.nan
    if 'Brake' not in ta.columns or 'Brake' not in tb.columns:
        return np.nan
    try:
        def brake_events(t):
            b = t['Brake'].astype(int)
            rising = b.diff() > 0
            # 保留 pandas Series（含 Timedelta 类型），避免 numpy timedelta64 无 .total_seconds()
            return list(t.loc[rising, 'SessionTime'])
        ea = brake_events(ta)
        eb = brake_events(tb)
        if len(ea) < 2 or len(eb) < 2:
            return np.nan
        diffs = []
        for t_a in ea:
            closest = min(eb, key=lambda x: abs(pd.Timedelta(x - t_a).total_seconds()))
            dt = abs(pd.Timedelta(t_a - closest).total_seconds()) * 1000
            if dt < 500:   # 500ms 窗口内才计入
                diffs.append(dt)
        return float(np.median(diffs)) if diffs else np.nan
    except Exception:
        return np.nan


# ── D5: TeamOrderConfidence ───────────────────────────────────
# 综合 Throttle_Drop_Index 与 Brake_Pressure_Delta 打分
def _calc_team_order_confidence(tdi: float, bpd: float) -> float:
    score = 0.0
    if not np.isnan(tdi):
        score += min(tdi / TEAM_ORDER_THROTTLE_THRESHOLD, 1.0) * 0.6
    if not np.isnan(bpd):
        score += min(bpd / TEAM_ORDER_BRAKE_THRESHOLD, 1.0) * 0.4
    return round(score, 3) if score > 0 else np.nan


# ── D6: Vmin_Deviation ───────────────────────────────────────
# 被超车手(B)在该圈黄旗区段最低速与参考圈同区段的偏差(%)
# 参考圈 = same driver 中 TrackStatus='1' 的中位圈
def _calc_vmin_deviation(driver_b: str, lap_n: int) -> float:
    tb = _get_tel(driver_b, lap_n)
    if tb is None or 'Speed' not in tb.columns:
        return np.nan
    # 黄旗时刻
    try:
        yellow_rows = laps_base[
            (laps_base['Driver'] == driver_b) &
            (laps_base['LapNumber'] == lap_n) &
            (laps_base['TrackStatus'].str.contains('2', na=False))
        ]
        if yellow_rows.empty:
            return np.nan
        # 参考圈
        ref_laps = laps_base[
            (laps_base['Driver'] == driver_b) &
            (laps_base['TrackStatus'] == '1') &
            (laps_base['IsAccurate'] == True)
        ]
        if ref_laps.empty:
            return np.nan
        ref_lap_n = int(ref_laps['LapNumber'].median())
        t_ref = _get_tel(driver_b, ref_lap_n)
        if t_ref is None or 'Speed' not in t_ref.columns:
            return np.nan
        if 'RelativeDistance' not in tb.columns or 'RelativeDistance' not in t_ref.columns:
            return np.nan
        # 黄旗区域用后 20% 作为占位（真实场景应映射时刻到距离）
        current_min = tb[tb['RelativeDistance'] > 0.7]['Speed'].min()
        ref_min     = t_ref[t_ref['RelativeDistance'] > 0.7]['Speed'].min()
        if pd.isna(ref_min) or ref_min < 1:
            return np.nan
        return round((current_min - ref_min) / ref_min * 100, 2)
    except Exception:
        return np.nan


# ── D7: OffTrack_Flag ────────────────────────────────────────
# pos_data.Status == 'OffTrack' 期间占比 > 5%
def _calc_offtrack_flag(driver_b: str, lap_n: int) -> bool:
    tb = _get_tel(driver_b, lap_n)
    if tb is None or 'Status' not in tb.columns:
        return False
    ratio = (tb['Status'] == 'OffTrack').mean()
    return bool(ratio > 0.05)


# ── D8: Lockup_Flag ──────────────────────────────────────────
# Brake==1 且 Throttle > 30 → 锁死代理
def _calc_lockup_flag(driver_b: str, lap_n: int) -> bool:
    tb = _get_tel(driver_b, lap_n)
    if tb is None:
        return False
    if 'Brake' not in tb.columns or 'Throttle' not in tb.columns:
        return False
    mask = (tb['Brake'].astype(int) == 1) & (tb['Throttle'] > 30)
    return bool(mask.sum() >= 2)   # 至少连续 2 个样本


# ── D9: LocalYellowFlag ──────────────────────────────────────
# 该圈 TrackStatus 含 '2'（局部黄旗）
def _calc_local_yellow(lap_n: int) -> bool:
    rows = laps_base[laps_base['LapNumber'] == lap_n]
    return bool(rows['TrackStatus'].str.contains('2', na=False).any())


# ── D10: OffTrackConfidence ──────────────────────────────────
def _calc_offtrack_confidence(vmin_dev, offtrack, lockup, yellow) -> float:
    score = 0.0
    if not np.isnan(vmin_dev) and vmin_dev > OFF_TRACK_VMIN_THRESHOLD:
        score += 0.4
    if offtrack:
        score += 0.3
    if lockup:
        score += 0.2
    if yellow:
        score += 0.1
    return round(score, 3) if score > 0 else np.nan


# ── 主循环：填充 full_overtakes 的所有特征列 ─────────────────
if not full_overtakes.empty:
    drs_list, reldist_list     = [], []
    tdi_list, bpd_list, toc_list = [], [], []
    vmin_list, ot_list, lk_list, ly_list, otc_list = [], [], [], [], []

    for _, row in full_overtakes.iterrows():
        a, b, ln = row['Overtaker'], row['Overtaken'], row['LapNumber']

        drs   = _calc_drs(a, ln)
        rd    = _calc_rel_dist(a, ln)
        tdi   = _calc_throttle_drop(a, b, ln)
        bpd   = _calc_brake_delta(a, b, ln)
        toc   = _calc_team_order_confidence(tdi, bpd)
        vmin  = _calc_vmin_deviation(b, ln)
        ot    = _calc_offtrack_flag(b, ln)
        lk    = _calc_lockup_flag(b, ln)
        ly    = _calc_local_yellow(ln)
        otc   = _calc_offtrack_confidence(vmin, ot, lk, ly)

        drs_list.append(drs);   reldist_list.append(rd)
        tdi_list.append(tdi);   bpd_list.append(bpd);   toc_list.append(toc)
        vmin_list.append(vmin); ot_list.append(ot);     lk_list.append(lk)
        ly_list.append(ly);     otc_list.append(otc)

    full_overtakes = full_overtakes.assign(
        Is_DRS_Active          = drs_list,
        RelativeDistance       = reldist_list,
        Throttle_Drop_Index    = tdi_list,
        Brake_Pressure_Delta   = bpd_list,
        TeamOrderConfidence    = toc_list,
        Vmin_Deviation         = vmin_list,
        OffTrack_Flag          = ot_list,
        Lockup_Flag            = lk_list,
        LocalYellowFlag        = ly_list,
        OffTrackConfidence     = otc_list,
    )
    print("Phase D 完成：全量特征填充完毕")
    print(full_overtakes[[
        'LapNumber', 'Overtaker', 'Overtaken',
        'Is_DRS_Active', 'RelativeDistance',
        'IsTeammate', 'Throttle_Drop_Index', 'TeamOrderConfidence',
        'OffTrack_Flag', 'Lockup_Flag', 'OffTrackConfidence'
    ]].to_string(index=False))
else:
    print("无候选超车，跳过特征计算。")


Phase D 完成：全量特征填充完毕
 LapNumber Overtaker Overtaken  Is_DRS_Active  RelativeDistance  IsTeammate  Throttle_Drop_Index  TeamOrderConfidence  OffTrack_Flag  Lockup_Flag  OffTrackConfidence
         2       SAR       RIC          False          0.488139          False             0.000000                0.400          False        False                 NaN
         3       NOR       ALO          False          0.489598          False             0.000000                0.133          False        False                 NaN
         3       RUS       LEC           True          0.488911          False             1.000000                0.460          False         True                 0.2
         5       PIA       ALO           True          0.488650          False             0.000000                0.400          False         True                 0.2
         5       STR       BOT          False          0.484961          False             1.000000                0.460          False   

In [15]:

# ============================================================
# Cell E — Phase E: 派生三张子表 + Schema 强制检查
# ============================================================

# ── 超集列顺序定义 ────────────────────────────────────────────
SUPERSET_COLS = [
    # 基础 10 列
    'LapNumber', 'Overtaker', 'OvertakerNumber', 'OvertakerTeam',
    'PositionBefore', 'PositionAfter', 'Overtaken', 'OvertakenNumber',
    'OvertakenTeam', 'SessionTime',
    # 审计
    'SameLapWindowStart', 'SameLapWindowEnd',
    'PitInfluenceFlag', 'DNFInfluenceFlag',
    # 研判
    'IsTeammate',
    'Is_DRS_Active', 'RelativeDistance',
    'Throttle_Drop_Index', 'Brake_Pressure_Delta', 'TeamOrderConfidence',
    'Vmin_Deviation', 'OffTrack_Flag', 'Lockup_Flag',
    'LocalYellowFlag', 'OffTrackConfidence',
]

def enforce_schema(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """确保 df 包含全部超集列，缺失列以 NaN 填充；返回标准列序的副本。"""
    for col in SUPERSET_COLS:
        if col not in df.columns:
            df = df.assign(**{col: np.nan})
    df = df[SUPERSET_COLS].copy()
    print(f"  [{name}] 行数={len(df)} | columns={len(df.columns)}")
    null_rate = df.isnull().mean().round(3)
    hi_null = null_rate[null_rate > 0.5]
    if not hi_null.empty:
        print(f"    ⚠ 高缺失率列(>50%): {hi_null.to_dict()}")
    return df


# ── E1: team_order_exchange ────────────────────────────────
# 条件：同一车队 AND TeamOrderConfidence ≥ 阈值
mask_team = (
    full_overtakes['IsTeammate'].eq(True) &
    full_overtakes['TeamOrderConfidence'].fillna(0.0).ge(TEAM_ORDER_CONFIDENCE_THRESHOLD)
)
team_order_exchange = enforce_schema(full_overtakes[mask_team].copy(), 'team_order_exchange')


# ── E2: off_track_advantage ────────────────────────────────
# 条件：OffTrack_Flag OR Lockup_Flag OR LocalYellowFlag
#       AND OffTrackConfidence ≥ 阈值
mask_offtrack = (
    (
        full_overtakes['OffTrack_Flag'].eq(True) |
        full_overtakes['Lockup_Flag'].eq(True)   |
        full_overtakes['LocalYellowFlag'].eq(True)
    ) &
    full_overtakes['OffTrackConfidence'].fillna(0.0).ge(OFF_TRACK_CONFIDENCE_THRESHOLD)
)
off_track_advantage = enforce_schema(full_overtakes[mask_offtrack].copy(), 'off_track_advantage')


# ── E3: clean_overtakes ────────────────────────────────────
# full_overtakes 中不属于 team_order 或 off_track 的行
# 用 (LapNumber, Overtaker, Overtaken) 三元组作为唯一键
_team_keys = set(zip(
    team_order_exchange['LapNumber'],
    team_order_exchange['Overtaker'],
    team_order_exchange['Overtaken'],
))
_ot_keys = set(zip(
    off_track_advantage['LapNumber'],
    off_track_advantage['Overtaker'],
    off_track_advantage['Overtaken'],
))
_dirty_keys = _team_keys | _ot_keys
mask_clean = ~pd.Series(
    list(zip(full_overtakes['LapNumber'],
             full_overtakes['Overtaker'],
             full_overtakes['Overtaken'])),
    index=full_overtakes.index
).isin(_dirty_keys)
clean_overtakes = enforce_schema(full_overtakes[mask_clean].copy(), 'clean_overtakes')

# ── E4: full_overtakes 也做 schema 规范化 ──────────────────
full_overtakes = enforce_schema(full_overtakes.copy(), 'full_overtakes')

print("\nPhase E 完成")
print(f"  full_overtakes      : {len(full_overtakes)} 行")
print(f"  team_order_exchange : {len(team_order_exchange)} 行")
print(f"  off_track_advantage : {len(off_track_advantage)} 行")
print(f"  clean_overtakes     : {len(clean_overtakes)} 行")


  [team_order_exchange] 行数=0 | columns=25
  [off_track_advantage] 行数=0 | columns=25
  [clean_overtakes] 行数=24 | columns=25
    ⚠ 高缺失率列(>50%): {'Vmin_Deviation': 1.0}
  [full_overtakes] 行数=24 | columns=25
    ⚠ 高缺失率列(>50%): {'Vmin_Deviation': 1.0}

Phase E 完成
  full_overtakes      : 24 行
  team_order_exchange : 0 行
  off_track_advantage : 0 行
  clean_overtakes     : 24 行


In [16]:

# ============================================================
# Cell F — Phase F: 验证输出 / 汇总
# ============================================================

print("=" * 60)
print(f"赛季/分站: {YEAR} R{ROUND}  ({SESSION_TYPE})")
print("=" * 60)

tables = {
    'full_overtakes'      : full_overtakes,
    'team_order_exchange' : team_order_exchange,
    'off_track_advantage' : off_track_advantage,
    'clean_overtakes'     : clean_overtakes,
}

for name, df in tables.items():
    print(f"\n── {name} ({len(df)} 行) ──")
    if df.empty:
        print("  (空)")
        continue
    # 关键列展示
    show_cols = [c for c in [
        'LapNumber', 'Overtaker', 'OvertakerTeam',
        'PositionBefore', 'PositionAfter',
        'Overtaken', 'OvertakenTeam',
        'IsTeammate', 'TeamOrderConfidence',
        'OffTrack_Flag', 'OffTrackConfidence',
        'Is_DRS_Active',
    ] if c in df.columns]
    print(df[show_cols].head(10).to_string(index=False))

# ── 特征覆盖率报告 ───────────────────────────────────────────
print("\n── full_overtakes 特征缺失率 ──")
feature_cols = [
    'Is_DRS_Active', 'RelativeDistance',
    'Throttle_Drop_Index', 'Brake_Pressure_Delta', 'TeamOrderConfidence',
    'Vmin_Deviation', 'OffTrack_Flag', 'Lockup_Flag',
    'LocalYellowFlag', 'OffTrackConfidence',
]
if not full_overtakes.empty:
    miss = full_overtakes[feature_cols].isnull().mean().mul(100).round(1)
    for col, pct in miss.items():
        flag = "  ⚠" if pct > 50 else ""
        print(f"  {col:30s}: {pct:5.1f}% 缺失{flag}")

print("\n✓ 全流程完成。four DataFrames 已就绪。")


赛季/分站: 2024 R1  (R)

── full_overtakes (24 行) ──
 LapNumber Overtaker   OvertakerTeam  PositionBefore  PositionAfter Overtaken OvertakenTeam  IsTeammate  TeamOrderConfidence  OffTrack_Flag  OffTrackConfidence  Is_DRS_Active
         2       SAR        Williams              15             14       RIC            RB          False                0.400          False                 NaN          False
         3       NOR         McLaren               7              6       ALO  Aston Martin          False                0.133          False                 NaN          False
         3       RUS        Mercedes               3              2       LEC       Ferrari          False                0.460          False                 0.2           True
         5       PIA         McLaren               8              7       ALO  Aston Martin          False                0.400          False                 0.2           True
         5       STR    Aston Martin              19            